1. What is Machine Learning?
Machine Learning (ML) is a subfield of Artificial Intelligence where algorithms learn patterns from data rather than being explicitly programmed with rules. Instead of writing if/else logic, you feed the model examples and it figures out the rules on its own.

Core idea: Given enough examples of inputs and outputs, a model can generalise to predict outputs for inputs it has never seen before.

The typical ML pipeline looks like this:
Raw Data → EDA → Preprocessing → Split → Train → Evaluate → Predict

2. The 3 Types of Machine Learning:
    Supervised Learning
    Trained on labelled data — every input has a known correct output. The model learns the mapping from input → output.

    → Linear Regression, Logistic Regression, SVM, Decision Trees

    Unsupervised Learning
    Trained on unlabelled data. The model discovers hidden structure or groupings on its own with no guidance.

    → K-Means Clustering, PCA, Autoencoders

    Reinforcement Learning
    An agent learns by taking actions in an environment, receiving rewards or penalties, and maximising long-term reward.

    → Q-Learning, Deep Q-Network (DQN), PPO

This notebook focuses on Supervised Learning — both Regression (predicting a number) and Classification (predicting a category).

3. What is EDA?
Exploratory Data Analysis (EDA) is the process of examining a dataset before building any model. 
You are trying to understand its structure, catch problems, and find patterns. EDA typically involves:
    Checking shape, dtypes, and column names
    Identifying missing values and duplicates
    Viewing summary statistics (mean, std, min, max)
    Visualising distributions and correlations
    Understanding which features are relevant to the target
    
EDA is not optional — it directly informs every preprocessing decision you make downstream.

4. Importing Libraries
These are the standard libraries used throughout every ML project. Import them at the top of every notebook.

 i. numpy
 ii. pandas
 iii. matplotlib
 iv. seaborn
 v. scikit-learn

In [18]:
# ─── Core data manipulation ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ─── Preprocessing ───────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ─── Models ──────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

# ─── Evaluation metrics ──────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score

print("All libraries imported successfully ✓")

All libraries imported successfully ✓


5. Loading & Exploring the Dataset (EDA)
We use the Iris dataset — 150 flower samples with 4 numeric features and a species label. It works for both regression and classification demos

In [19]:
from sklearn.datasets import load_iris

# Load dataset into a pandas DataFrame
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target_names[iris.target]  # string labels

# ── Basic EDA ────────────────────────────────────────────────────────────────
print("Shape:", df.shape)            # rows, columns
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
print(df.describe())

Shape: (150, 5)

Column names:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'species']

First 5 rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

  species  
0  setosa  
1  setosa  
2  setosa  
3  setosa  
4  setosa  

Data types:
sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
species                  str
dtype: object

Summary statistics:
       sepal length (cm)  sepal width (cm)  petal length (cm)  \
count         150.000000        150.000000         150.

Checking for missing values

In [20]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
species              0
dtype: int64

Total missing values: 0
Duplicate rows: 1


6. Cleaning — Removing Duplicates & Missing Values
Raw data is almost never clean. You must handle these two problems before training anything:
    Duplicates — identical rows that bias the model toward over-represented examples
    Missing values (NaN) — cells with no value; most algorithms cannot handle them at all

In [21]:
# ── Remove duplicate rows ────────────────────────────────────────────────────
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

# ── Handle missing values ────────────────────────────────────────────────────
# Strategy 1: Drop rows with any NaN
df_dropped = df.dropna()

# Strategy 2: Fill numeric NaNs with column mean (more common — preserves data)
df_filled = df.copy()
numeric_cols = df_filled.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    df_filled[col].fillna(df_filled[col].mean(), inplace=True)

# Strategy 3: Fill categorical NaNs with the most frequent value (mode)
cat_cols = df_filled.select_dtypes(include=['object']).columns
for col in cat_cols:
    df_filled[col].fillna(df_filled[col].mode()[0], inplace=True)

print("Missing values remaining:", df_filled.isnull().sum().sum())
df = df_filled  # use the cleaned version going forward

After removing duplicates: (149, 5)
Missing values remaining: 0


C:\Users\danny\AppData\Local\Temp\ipykernel_5812\3971057878.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_filled[col].fillna(df_filled[col].mean(), inplace=True)
C:\Users\danny\AppData\Local\Temp\ipykernel_5812\3971057878.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'obj

LinearRegression and LogisiticRegression

RandomForestClassification

Bagging

Boosting (XGBoost)

DeepLearning (ANN, CNN, RNN)

Layers of Neural Networks:
    1. Convolution Layer - 2D (32 Layers and 64 Layers)
    2. MaxPooling Layer - 2D (2 Layers)
    3. Flatten Layer
    4. Dropout Layer
    5. Dense Layer (128 Neurons and 10 Neurons)